[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanemat/uol-fp/blob/main/proto1/pipeline.ipynb)

# Methodology Extraction Pipeline

Extracts structured methodology from a research paper.

Output format:
```json
{
  "Design": "experiment",
  "Method": ["BERT"],
  "Data": ["MNIST"],
  "Evaluation": ["accuracy"]
}
```

**Run in order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7**

## Setup

In [ ]:
!pip install requests

In [ ]:
import json
import re
from dataclasses import dataclass, field
from enum import Enum

print("Setup complete.")

## Data Models

In [ ]:
class DesignType(str, Enum):
    EXPERIMENT = "experiment"
    SURVEY = "survey"
    CASE_STUDY = "case_study"
    THEORETICAL = "theoretical"
    ALGORITHM_DEVELOPMENT = "algorithm_development"
    UNKNOWN = "unknown"


@dataclass
class CandidateWithContext:
    candidate: str
    sentence: str
    section: str = "unknown"
    source_paper: str = ""


@dataclass
class MethodologyProfile:
    design: DesignType = DesignType.UNKNOWN
    method: list = field(default_factory=list)
    task: list = field(default_factory=list)
    data: list = field(default_factory=list)
    evaluation: list = field(default_factory=list)

    def to_dict(self):
        result = {
            "Design": self.design.value,
            "Method": self.method,
            "Task": self.task,
        }
        optional = {}
        if self.data:
            optional["Data"] = self.data
        if self.evaluation:
            optional["Evaluation"] = self.evaluation
        if optional:
            result["Optional"] = optional
        return result


print("Models ready.")

## Step 0 — Load TEI XML

Upload a TEI XML file produced by local GROBID (`python pdf_to_xml.py paper.pdf`).

In [ ]:
from xml.etree import ElementTree as ET

from google.colab import files

NS = {"tei": "http://www.tei-c.org/ns/1.0"}
SKIP_HEADINGS = {"references", "acknowledgements", "acknowledgments"}


def _text(element) -> str:
    return " ".join(element.itertext()).strip()


uploaded = files.upload()
xml_filename = next(iter(uploaded))
xml_bytes = uploaded[xml_filename]

root = ET.fromstring(xml_bytes.decode("utf-8"))

abstract_el = root.find(".//tei:abstract", NS)
abstract_text = _text(abstract_el) if abstract_el is not None else ""

sections = []
for div in root.findall(".//tei:body//tei:div", NS):
    heading = div.findtext("tei:head", namespaces=NS) or ""
    if heading.lower().strip() in SKIP_HEADINGS:
        continue
    body = " ".join(_text(p) for p in div.findall("tei:p", NS)).strip()
    if body:
        sections.append({"heading": heading, "text": body})

if abstract_text:
    sections.insert(0, {"heading": "Abstract", "text": abstract_text})

print(f"Loaded : {xml_filename}")
print(f"Sections: {len(sections)}")
for s in sections:
    print(f"  - {s['heading']}")

## Step 1 — Candidate Extraction (SciBERT NER)

In [ ]:
_STOP = {
    "We",
    "Our",
    "The",
    "This",
    "In",
    "A",
    "An",
    "To",
    "For",
    "On",
    "Is",
    "It",
    "At",
    "By",
    "As",
    "Of",
    "Be",
    "Are",
    "Was",
    "Has",
    "Have",
    "From",
    "With",
    "That",
    "Which",
    "These",
    "Those",
    "Also",
    "Such",
    "Both",
    "Each",
}

_PATTERNS: list[tuple[str, int]] = [
    (r"\b[A-Z][A-Za-z0-9]*(?:-[A-Za-z0-9]+)*\b", 0),
    (r"\b[a-z]+-\d+\b", 0),
    (
        r"\b(?:accuracy|f1|precision|recall|bleu|rouge|auc"
        r"|attention|translation|summarization|classification|recognition|detection|generation)\b",
        re.IGNORECASE,
    ),
    (r"\b(?:dataset|corpus|benchmark)\b", re.IGNORECASE),
]


def extract_candidates(sections: list[dict]) -> list[CandidateWithContext]:
    results: list[CandidateWithContext] = []
    seen: set[str] = set()
    for section in sections:
        heading = section["heading"]
        sentences = [
            s.strip() for s in re.split(r"(?<=[.!?])\s+", section["text"]) if s.strip()
        ]
        for sent in sentences:
            for pattern, flags in _PATTERNS:
                for match in re.finditer(pattern, sent, flags):
                    term = match.group().strip()
                    if (
                        len(term) < 3
                        or len(term) > 40
                        or term in _STOP
                        or term.lower() in {s.lower() for s in _STOP}
                        or term in seen
                    ):
                        continue
                    seen.add(term)
                    results.append(
                        CandidateWithContext(
                            candidate=term, sentence=sent, section=heading
                        )
                    )
    return results


candidates_with_ctx = extract_candidates(sections)
print(f"Candidates: {len(candidates_with_ctx)}")

## Step 2 — Role Classification

In [ ]:
class Role(str, Enum):
    METHOD = "Method"
    TASK = "Task"
    DATA = "Data"
    EVALUATION = "Evaluation"
    OTHER = "Other"


_KNOWN_METHODS = {
    "bert",
    "gpt",
    "roberta",
    "xlnet",
    "t5",
    "gpt-2",
    "gpt-3",
    "lstm",
    "cnn",
    "rnn",
    "transformer",
    "attention",
    "svm",
    "random forest",
    "k-means",
    "resnet",
    "vgg",
    "scibert",
}
_KNOWN_TASKS = {
    "question answering",
    "text classification",
    "image classification",
    "named entity recognition",
    "machine translation",
    "summarization",
    "sentiment analysis",
    "relation extraction",
    "coreference resolution",
}
_KNOWN_DATA = {
    "mnist",
    "cifar",
    "squad",
    "glue",
    "superglue",
    "imagenet",
    "sst-2",
    "sst-1",
    "conll",
    "wikitext",
    "bookcorpus",
    "imdb",
    "yelp",
    "amazon",
    "snli",
    "mnli",
}
_KNOWN_EVAL = {
    "accuracy",
    "f1",
    "precision",
    "recall",
    "bleu",
    "rouge",
    "auc",
    "map",
    "ndcg",
    "perplexity",
    "em",
    "exact match",
}

_METHOD_RE = re.compile(
    r"\b(neural|network|model|algorithm|architecture|classifier|proposed)\b",
    re.IGNORECASE,
)
_TASK_RE = re.compile(
    r"\b(task|problem|classification|recognition|detection|generation|translation)\b",
    re.IGNORECASE,
)
_DATA_RE = re.compile(
    r"\b(dataset|corpus|benchmark|collection|training|test)\b",
    re.IGNORECASE,
)
_EVAL_RE = re.compile(
    r"\b(score|metric|performance|rate|result)\b",
    re.IGNORECASE,
)


def classify_role(candidate: str, context: str = "") -> Role:
    low = candidate.lower()
    if low in _KNOWN_EVAL or _EVAL_RE.search(low):
        return Role.EVALUATION
    if low in _KNOWN_DATA or _DATA_RE.search(context.lower()):
        return Role.DATA
    if low in _KNOWN_TASKS or _TASK_RE.search(context.lower()):
        return Role.TASK
    if low in _KNOWN_METHODS or _METHOD_RE.search(context.lower()):
        return Role.METHOD
    return Role.OTHER


classified: dict[str, Role] = {}
for cwc in candidates_with_ctx:
    classified[cwc.candidate] = classify_role(cwc.candidate, cwc.sentence)

for term, role in sorted(classified.items()):
    print(f"  {role.value:12s}  {term}")

## Candidate Log (Error Analysis)


In [ ]:
log = []
for cwc in candidates_with_ctx:
    log.append(
        {
            "candidate": cwc.candidate,
            "role": classified.get(cwc.candidate, Role.OTHER).value,
            "sentence": cwc.sentence,
            "section": cwc.section,
        }
    )

print(json.dumps(log, indent=2))

## Step 3 — Design Detection

In [ ]:
_DESIGN_PATTERNS: list[tuple[DesignType, list[str]]] = [
    (
        DesignType.EXPERIMENT,
        [r"\bexperiment\w*\b", r"\buser study\b", r"\bablation\b"],
    ),
    (
        DesignType.SURVEY,
        [r"\bsurvey\b", r"\bliterature review\b", r"\bsystematic review\b"],
    ),
    (DesignType.CASE_STUDY, [r"\bcase study\b", r"\bcase studies\b"]),
    (DesignType.THEORETICAL, [r"\btheor\w+\b", r"\bproof\b", r"\bformal\w*\b"]),
    (
        DesignType.ALGORITHM_DEVELOPMENT,
        [r"\balgorithm\w*\b", r"\barchitecture\b", r"\bpropose\w*\b", r"\bnovel\b"],
    ),
]


def detect_design(text: str) -> DesignType:
    text_lower = text.lower()
    scores: dict[DesignType, int] = {}
    for design_type, patterns in _DESIGN_PATTERNS:
        count = sum(len(re.findall(p, text_lower)) for p in patterns)
        if count > 0:
            scores[design_type] = count
    if not scores:
        return DesignType.UNKNOWN
    return max(scores, key=lambda k: scores[k])


full_text = " ".join(s["text"] for s in sections)
design = detect_design(full_text)
print(f"Design: {design.value}")

## Step 4 — Build JSON Output

In [ ]:
profile = MethodologyProfile(
    design=design,
    method=[t for t, r in classified.items() if r == Role.METHOD],
    task=[t for t, r in classified.items() if r == Role.TASK],
    data=[t for t, r in classified.items() if r == Role.DATA],
    evaluation=[t for t, r in classified.items() if r == Role.EVALUATION],
)

print(json.dumps(profile.to_dict(), indent=2))

## Step 5 — Consistency Checking

In [ ]:
@dataclass
class ConsistencyResult:
    is_valid: bool
    warnings: list


def check_consistency(profile: MethodologyProfile) -> ConsistencyResult:
    warnings = []
    if profile.design == DesignType.EXPERIMENT:
        if not profile.task:
            warnings.append("Experimental paper without Task is weak.")
        if not profile.method:
            warnings.append("Experimental paper without Method is weak.")
    if profile.method and not profile.task:
        warnings.append("Method without Task may be incomplete.")
    if profile.design == DesignType.THEORETICAL and profile.evaluation:
        warnings.append("Theoretical design with Evaluation may be a mismatch.")
    return ConsistencyResult(is_valid=len(warnings) == 0, warnings=warnings)


result = check_consistency(profile)
print(f"Valid: {result.is_valid}")
for w in result.warnings:
    print(f"  WARNING: {w}")
if result.is_valid:
    print("  No issues found.")